# 02 · Limpieza y unificación de datos

**Objetivo:** pasar de tres archivos CSV sueltos y a medio rellenar a **una sola tabla lista para analizar**.

Al terminar sabrás:

1. Qué hace la función `cargar()` y por qué convierte `PENDIENTE` en `NaN`.
2. Qué es `groupby()` y por qué hace la media de varios precios por año.
3. Qué significa **deflactar** (pasar a precios constantes con el IPC).
4. Unir las series en `data/processed/serie_unificada.csv`.

## 0 · Imports y acceso a `src/`

Las funciones de limpieza viven en `src/limpieza.py` (se escribieron en la Fase 1). Aquí solo las importamos: **no reescribimos código que ya existe**.

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd()
if not (RAIZ / 'data').exists():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

from src import limpieza

print('Proyecto:', RAIZ)

## 1 · Cargar los tres archivos

`limpieza.cargar()` hace dos cosas:

1. Lee el CSV.
2. Convierte `PENDIENTE` → `NaN` y las columnas numéricas → número (`pd.to_numeric`).

Es el mismo proceso que aprendiste en el notebook 01, pero **empaquetado en una función** para no repetirlo.

In [ ]:
salarios = limpieza.cargar('salarios_cuba.csv')
precios = limpieza.cargar('precios_paneles.csv')
ipc = limpieza.cargar('ipc_cuba.csv')

salarios

In [ ]:
print('--- Cuántos datos faltan en salarios ---')
salarios.isna().sum()

### Pregunta de control

Antes de seguir, responde (sin mirar la respuesta): si `salario_medio_cup` tiene `10` valores NaN, ¿qué significa?

*Respuesta:* que ninguna de las 10 filas tiene dato todavía. Si dice `3`, faltan 3 años por rellenar.

## 2 · Los precios: uno por año no, varios

En un mismo año aparecerán **varios anuncios** de paneles (TiendML, Revolico, etc.), cada uno con su precio. Para comparar año con año necesitamos **un solo número por año**: su media.

Eso hace `groupby()`: agrupa las filas por año, calcula un resumen y devuelve una fila por grupo.

In [ ]:
precios.groupby('anio')['precio_cup'].mean()

> **Advertencia metodológica:** la media de *todos* los anuncios puede no ser representativa
> (un anuncio de 10 paneles usados abarata la media). Por eso guardamos también `potencia_wp`
> y, en la Fase 5, convendrá usar **precio por vatio (CUP/Wp)** en vez de precio absoluto.
> Lo ideal sería filtrar solo anuncios nuevos y comparables.

## 3 · Deflactar: ¿por qué el IPC?

1000 pesos de 2016 **no compran lo mismo** que 1000 pesos de 2025. Comparar salarios o precios nominales a lo largo del tiempo es una trampa clásica.

La solución es el **IPC** (*Índice de Precios al Consumidor*): mide cuánto han subido los precios en general.

```
valor real (año base) = valor nominal × IPC(año base) / IPC(año concreto)
```

Al resultado se le llama **precios constantes**: quitamos la inflación de la cuenta para ver si *realmente* algo subió o bajó.

In [ ]:
ejemplo = salarios.copy()
ejemplo['salario_real'] = limpieza.a_precios_constantes(salarios, 'salario_medio_cup')

print('Año base usado por el módulo:', limpieza.ANIO_BASE)
ejemplo[['anio', 'salario_medio_cup', 'salario_real']]

Con el IPC en `NaN` todo sale `NaN`: la fórmula funciona, pero le faltan los datos.
Rellena `ipc_cuba.csv` y vuelve a ejecutar esta celda.

## 4 · Unificar: una tabla, todos los años

`limpieza.unificar()` hace un **merge** (unión por la columna `anio`) de las tres tablas con `how='outer'`: se conservan todos los años aunque falte algo en alguna serie. Así no perdemos filas.

In [ ]:
serie = limpieza.unificar()
serie

In [ ]:
serie.info()

## 5 · Guardar el resultado

El archivo generado queda en `data/processed/`. **Nunca se modifica `data/raw/`**: los datos crudos son la evidencia y se conservan tal cual se recibieron.

In [ ]:
ruta = limpieza.guardar(serie)
print('Guardado en:', ruta)
print(ruta.read_text(encoding='utf-8'))

## 6 · Estado del proyecto

| Archivo | Estado |
|---|---|
| `salarios_cuba.csv` | ⏳ pendiente de tus datos del ONEI |
| `ipc_cuba.csv` | ⏳ pendiente (ONEI, índice de precios) |
| `precios_paneles.csv` | ⏳ pendiente (scraping, Fase 3) |

**Tu tarea:**

1. Rellenar `salarios_cuba.csv` (puedes pegarme las cifras y yo te devuelvo el CSV).
2. Rellenar `ipc_cuba.csv` con el índice de precios del mismo anuario.
3. Reejecutar este notebook de arriba a abajo y comprobar que ya no hay `NaN` en las columnas clave.

## Glosario

| Término | Significado |
|---|---|
| **Deflactar** | Quitar el efecto de la inflación a una cifra |
| **IPC** | Índice de Precios al Consumidor: cuánto suben los precios |
| **Valor nominal** | El precio tal cual se anunció en ese año |
| **Valor real / constante** | El valor expresado en dinero de un año base |
| **`groupby()`** | Agrupa filas por una columna y resume cada grupo |
| **`merge` / `join`** | Une dos tablas por una columna en común |